# Notebook 00 — Environment Setup and Training-Data Generation

This notebook verifies that every dependency is importable, configures the NVIDIA cuQuantum backend, and generates the 50 000-sample `(features, labels)` corpus used to train the QED scheduler (§4.2 of the paper).  
Run this notebook **first** before any other notebook in the series.

**Outputs written to disk**
- `data/training/features.csv` — six scalar features per compiled circuit  
- `data/training/labels.csv`  — target columns `delta_S` and `retention`

In [ ]:
# ── stdlib ──────────────────────────────────────────────────────────────────
import os, sys, time, warnings, importlib
from pathlib import Path

warnings.filterwarnings('ignore')

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

DATA_DIR = ROOT / 'data' / 'training'
DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f'Project root : {ROOT}')
print(f'Training dir : {DATA_DIR}')

In [ ]:
# ── dependency check ─────────────────────────────────────────────────────────
REQUIRED = [
    'qiskit', 'qiskit_aer', 'numpy', 'scipy', 'pandas',
    'xgboost', 'sklearn', 'matplotlib', 'seaborn', 'tqdm',
]
OPTIONAL = ['cuquantum', 'custatevec']

missing = []
for pkg in REQUIRED:
    try:
        importlib.import_module(pkg)
        print(f'  ✓  {pkg}')
    except ImportError:
        print(f'  ✗  {pkg}  ← MISSING')
        missing.append(pkg)

for pkg in OPTIONAL:
    try:
        importlib.import_module(pkg)
        print(f'  ✓  {pkg}  (optional, GPU acceleration enabled)')
    except ImportError:
        print(f'  ~  {pkg}  (optional, not found — CPU fallback active)')

if missing:
    raise EnvironmentError(f'Missing required packages: {missing}. Run: pip install {" ".join(missing)}')

In [ ]:
# ── version banner ───────────────────────────────────────────────────────────
import qiskit, numpy as np, pandas as pd, xgboost as xgb

print('Library versions')
print(f'  qiskit   : {qiskit.__version__}')
print(f'  numpy    : {np.__version__}')
print(f'  pandas   : {pd.__version__}')
print(f'  xgboost  : {xgb.__version__}')

In [ ]:
# ── GPU / cuQuantum probe ─────────────────────────────────────────────────────
GPU_AVAILABLE = False
try:
    import cuquantum  # noqa: F401
    GPU_AVAILABLE = True
    print('cuQuantum SDK detected — density-matrix simulation will use GPU.')
except ImportError:
    pass

try:
    import subprocess
    result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                             '--format=csv,noheader'], capture_output=True, text=True, timeout=5)
    if result.returncode == 0:
        for line in result.stdout.strip().splitlines():
            print(f'  GPU: {line}')
except Exception:
    print('nvidia-smi not available — assuming CPU-only environment.')

In [ ]:
# ── noise-model loader ────────────────────────────────────────────────────────
import json
from src.simulation.noise_models import (
    build_superconducting_noise_model,
    build_trapped_ion_noise_model,
    build_adversarial_noise_model,
)

NOISE_PROFILES = {
    'superconducting': build_superconducting_noise_model(),
    'trapped_ion':     build_trapped_ion_noise_model(),
    'adversarial':     build_adversarial_noise_model(),
}
print(f'Loaded {len(NOISE_PROFILES)} noise profiles: {list(NOISE_PROFILES.keys())}')

In [ ]:
# ── random-circuit generator ──────────────────────────────────────────────────
from qiskit.circuit.random import random_circuit
from qiskit import transpile
from qiskit_aer import AerSimulator
from src.compiler.mapping_pass import HardwareAwareMappingPass
from src.scheduler.feature_extractor import extract_features


def generate_random_circuit(n_qubits: int, depth: int, seed: int):
    """Return a random QuantumCircuit suitable for compilation experiments."""
    rng = np.random.default_rng(seed)
    qc = random_circuit(n_qubits, depth, max_operands=2,
                        measure=False, seed=int(rng.integers(1 << 31)))
    return qc


print('random_circuit generator ready.')

In [ ]:
# ── main corpus generation loop ───────────────────────────────────────────────
# Paper §4.2: 50 000 circuit-noise pairs; 10k validation, 10k held-out test.
# For demonstration the default N_SAMPLES is reduced to 1 000;
# set N_SAMPLES = 50_000 for full production run.

from tqdm.auto import tqdm
from src.simulation.simulator import run_noisy_simulation
from src.qed.circuit_builder import build_qed_circuit
from src.metrics.aggregator import compute_success_probability, compute_retention

N_SAMPLES   = int(os.environ.get('N_SAMPLES', 1_000))
QUBIT_RANGE = (6, 12)
DEPTH_RANGE = (10, 80)
FREQ_CANDS  = [0.0, 0.25, 0.50, 0.75, 1.0]
POS_CANDS   = [0, 1, 2, 3]
RNG_SEED    = 42
rng = np.random.default_rng(RNG_SEED)

feature_rows = []
label_rows   = []

noise_names = list(NOISE_PROFILES.keys())

for i in tqdm(range(N_SAMPLES), desc='Generating corpus'):
    n_q   = int(rng.integers(*QUBIT_RANGE, endpoint=True))
    depth = int(rng.integers(*DEPTH_RANGE, endpoint=True))
    noise_name = noise_names[i % len(noise_names)]
    noise_model = NOISE_PROFILES[noise_name]
    freq  = FREQ_CANDS[i % len(FREQ_CANDS)]
    pos   = POS_CANDS[i  % len(POS_CANDS)]

    # Generate & compile baseline
    qc = generate_random_circuit(n_q, depth, seed=i)
    try:
        mapper   = HardwareAwareMappingPass(n_qubits=n_q)
        compiled  = mapper.run(qc)
        feats    = extract_features(compiled, n_qubits=n_q)

        # Baseline simulation
        counts_base = run_noisy_simulation(compiled, noise_model, shots=512)
        s_base      = compute_success_probability(counts_base)

        # QED circuit
        qed_circ    = build_qed_circuit(compiled, syndrome_freq=freq, insertion_pos=pos)
        counts_qed  = run_noisy_simulation(qed_circ, noise_model, shots=512)
        s_qed       = compute_success_probability(counts_qed)
        retention   = compute_retention(counts_qed, shots=512)

        feats.update({'freq': freq, 'pos': pos, 'noise_profile': noise_name})
        feature_rows.append(feats)
        label_rows.append({'delta_S': s_qed - s_base, 'retention': retention})
    except Exception as exc:
        # Skip circuits that fail to compile; log count at end
        label_rows.append({'delta_S': float('nan'), 'retention': float('nan')})
        feature_rows.append({'error': str(exc)})

print(f'\nCorpus generation complete: {N_SAMPLES} samples attempted.')

In [ ]:
# ── persist to CSV ────────────────────────────────────────────────────────────
df_feat   = pd.DataFrame(feature_rows)
df_labels = pd.DataFrame(label_rows)

# Drop rows where either circuit failed to compile
valid_mask = df_labels['delta_S'].notna() & df_feat.get('error', pd.Series(index=df_feat.index)).isna()
if 'error' in df_feat.columns:
    valid_mask = df_labels['delta_S'].notna()
    df_feat  = df_feat.drop(columns=['error'], errors='ignore')

df_feat   = df_feat[valid_mask].reset_index(drop=True)
df_labels = df_labels[valid_mask].reset_index(drop=True)

feat_path   = DATA_DIR / 'features.csv'
labels_path = DATA_DIR / 'labels.csv'

df_feat.to_csv(feat_path,   index=False)
df_labels.to_csv(labels_path, index=False)

print(f'features.csv → {feat_path}  ({len(df_feat)} rows)')
print(f'labels.csv   → {labels_path}  ({len(df_labels)} rows)')
df_feat.head(3)

In [ ]:
# ── descriptive statistics ────────────────────────────────────────────────────
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df_labels['delta_S'].hist(bins=40, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title(r'Distribution of $\Delta S$ (success uplift)')
axes[0].set_xlabel(r'$\Delta S$')
axes[0].set_ylabel('Count')

df_labels['retention'].hist(bins=40, ax=axes[1], color='darkorange', edgecolor='white')
axes[1].set_title('Distribution of Retention $R$')
axes[1].set_xlabel('Retention')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.savefig(DATA_DIR / 'corpus_distributions.png', dpi=150)
plt.show()
print('Corpus distributions saved.')

In [ ]:
# ── summary ───────────────────────────────────────────────────────────────────
print('\n=== Corpus summary ===')
print(df_labels.describe().round(4))
print('\nSetup complete — proceed to Notebook 01.')